# Session 3: Vectors in the wild

### From spray patterns to plane landings to AI

Vectors are one of the most used ideas in the modern world. Anything with a **size and a direction**, and anything you can describe with a **list of numbers**, is a vector.

- **Games:** every character's position, every velocity, every recoil kick and the direction the camera faces is a vector. A game engine adds and scales thousands of them, 60 or more times a second. That's exactly what you did in session 2.
- **Physics and engineering:** the forces on a bridge, a rocket's thrust, the wind pushing a plane. Velocity, force and displacement are vectors in GCSE Physics too.
- **Films and graphics:** every 3D model, including every character in an animated film, is built from thousands of points stored as vectors. Moving the camera means moving all of them.
- **Navigation:** GPS, drones and self-driving cars all work out where they are and where they're heading with vectors.
- **AI:** models like ChatGPT turn words, images and songs into vectors with hundreds or thousands of numbers each. Nobody types those numbers in: the model **learns** them while it trains, which is more like growing something than programming it. Things with similar meanings end up close together, and you can even do arithmetic with them.

Today you'll see the first two up close: a spray pattern like the one in Counter-Strike, and a plane landing in a storm. Later we'll get to AI. Every one of them uses only what you learned in sessions 1 and 2.

## Part 1: Spray control

In Counter-Strike, every gun has a **recoil pattern**. Each shot kicks your aim a little, always in the same order: an AK climbs up, then swings left, then right. Good players memorise the pattern and pull their mouse the opposite way while they spray.

Every kick is a vector. The kicks add up tip to tail, and pulling the mouse the opposite way means adding their **negatives**. That's session 2, at 10 shots a second.

### Opening the Spray Lab

The Spray Lab is a web page in this folder called `03-spray-lab.html`. Open it in either of these ways:

- **Option A:** run the cell below. It opens the page in your web browser.
- **Option B:** in File Explorer, go to the folder where you cloned this repo and double-click `03-spray-lab.html`. If it opens in Notepad instead, right-click it and choose **Open with → Chrome** (or Edge, or Firefox).

Keep the page open next to this notebook.

In [ ]:
import webbrowser
from pathlib import Path

webbrowser.open(Path("03-spray-lab.html").resolve().as_uri())

### Before you play

**Slow motion** is switched on to start with. Click **Show a perfect spray** and watch the two panels at the top: **Gun kick** on the left and **Your adjustment** on the right.

**Before you click:** a perfect spray keeps every bullet in the same hole. What do you think the green vector will be each time, compared with the red one?

> ### What the game is actually doing
>
> The gun has a list of kicks $\mathbf{k}_1, \mathbf{k}_2, \mathbf{k}_3, \dots$, one per shot. After $n$ shots, the total recoil is
>
> $$\mathbf{R} = \mathbf{k}_1 + \mathbf{k}_2 + \dots + \mathbf{k}_n$$
>
> and the bullet lands at **where you're aiming + $\mathbf{R}$**.
>
> - To keep every bullet in the same place, your total mouse movement has to be $-\mathbf{R}$, so that the total drift is $\mathbf{R} + (-\mathbf{R}) = \mathbf{0}$.
> - The **Recoil strength** slider multiplies every kick by the same number $k$. That's scalar multiplication, so the total recoil becomes $k\mathbf{R}$.
>
> Real games also add a little random spread on top, which is why even pros don't land every bullet in exactly the same hole.

### Now change the game

The recoil pattern is just a list of vectors in the code. Open `03-spray-lab.html` in a text editor (Notepad works; VS Code is nicer) and find the block that starts with `// ✏️ Change these`. Each kick is written as `[across, up]`.

1. Look at the row marked `// climbs up`: the 9 kicks after the first shot. Add them up on paper. Set **Recoil strength** to 1, spray exactly 10 shots in slow motion, and check your answer in the **total so far** box.
2. **The dream gun:** change the kicks so that the total recoil after 30 shots is $\mathbf{0}$, but the gun still kicks on every shot. Save the file, refresh the page, and try it. What does a perfect spray look like now?
3. Design a pattern that makes the red chain draw a shape: a square, a zigzag, or your initials.

### Check yourself (no computer)

1. A gun's first four kicks are $\begin{pmatrix} 0 \\ 0 \end{pmatrix}, \begin{pmatrix} 1 \\ 4 \end{pmatrix}, \begin{pmatrix} -2 \\ 5 \end{pmatrix}, \begin{pmatrix} 3 \\ 3 \end{pmatrix}$. What is the total recoil after four shots? What total mouse movement would cancel it?
2. At strength 1, a spray's total recoil is $\begin{pmatrix} -36 \\ 102 \end{pmatrix}$. What is it at strength 2?
3. In one spray, the total recoil was $\begin{pmatrix} -54 \\ 153 \end{pmatrix}$ and your mouse moved $\begin{pmatrix} 30 \\ -100 \end{pmatrix}$ in total. Find the total drift and its length.

## Part 2: Landing in a crosswind

When a strong wind blows **across** the runway, planes come in to land pointing to one side. They look like they're flying sideways, but they move straight down the runway. Pilots call this **crabbing**.

**Watch first:** search YouTube for [crosswind landings](https://www.youtube.com/results?search_query=crosswind+landing+crab+storm) and watch a minute or two of any storm-landing compilation. Look at which way each plane's nose points, and which way the plane actually moves.

The explanation is vector addition. A plane flies through the **air**, but the air itself is moving (that's the wind). So:

$$\text{velocity over the ground} = \text{velocity through the air} + \text{wind}$$

The animation shows the plane from above, with the vectors at the top. Speeds are in **knots**, the unit pilots use. First the plane points straight down the runway, then the pilot turns the nose into the wind.

**Before you run it:** if the nose points straight down the runway and the wind blows across it, where will the plane end up?

Run this cell first. It sets up manim for the rest of the notebook.

In [ ]:
from manim import *

config.media_embed = True

In [ ]:
%%manim -ql -v WARNING --progress_bar none Crosswind
import math

# ✏️ Change these
AIRSPEED = 130  # how fast the plane flies through the air, in knots
WIND = 50       # how fast the wind blows across the runway, in knots


def fmt(n):
    # 120.0 -> "120", -49.5 -> "−49.5"
    return f"{round(n, 1):g}".replace("-", "−")


def column(x, y, color, size=30):
    # A column vector drawn with text, so no LaTeX is needed
    top = Text(fmt(x), font_size=size, color=color)
    bottom = Text(fmt(y), font_size=size, color=color)
    entries = VGroup(top, bottom).arrange(DOWN, buff=0.18)
    left = Text("(", font_size=size).stretch_to_fit_height(entries.height * 1.3).next_to(entries, LEFT, buff=0.1)
    right = Text(")", font_size=size).stretch_to_fit_height(entries.height * 1.3).next_to(entries, RIGHT, buff=0.1)
    return VGroup(left, entries, right)


def plane_shape():
    # A plane seen from above, pointing right
    wings = Polygon([0.15, 0, 0], [-0.15, 0.55, 0], [-0.3, 0.55, 0], [-0.15, 0, 0],
                    [-0.3, -0.55, 0], [-0.15, -0.55, 0], stroke_width=0, fill_color=GREY_B, fill_opacity=1)
    tail = Polygon([-0.4, 0, 0], [-0.55, 0.22, 0], [-0.62, 0.22, 0], [-0.52, 0, 0],
                   [-0.62, -0.22, 0], [-0.55, -0.22, 0], stroke_width=0, fill_color=GREY_B, fill_opacity=1)
    body = Ellipse(width=1.1, height=0.18, stroke_width=0, fill_color=WHITE, fill_opacity=1)
    return VGroup(wings, tail, body)


class Crosswind(Scene):
    def construct(self):
        if WIND >= AIRSPEED:
            self.add(Text("The wind is at least as fast as the plane.\nIt can't fly along the runway at all!", font_size=30))
            self.wait()
            return

        SCALE = 1 / 30  # 30 knots = 1 square on the screen
        wind = np.array([0, -WIND, 0])

        # The runway from above, and some wind blowing down across it
        runway_y = -2.6
        start = np.array([-6, runway_y, 0])
        self.add(Rectangle(width=9, height=0.9, stroke_width=0, fill_color=GREY_E, fill_opacity=1).move_to([2.5, runway_y, 0]),
                 DashedLine([-2, runway_y, 0], [7, runway_y, 0], color=WHITE, dash_length=0.25))
        for x in (-3, -0.5, 2, 4.5):
            self.add(Arrow([x, -0.6, 0], [x, -1.5, 0], buff=0, color=RED, stroke_width=3).set_opacity(0.5))

        # The vector diagram, tip to tail: air (green), then wind (red), then the total (yellow)
        origin = np.array([-6.2, 1.4, 0])

        def diagram(air):
            ground = air + wind
            return VGroup(Arrow(origin, origin + air * SCALE, buff=0, color=GREEN),
                          Arrow(origin + air * SCALE, origin + ground * SCALE, buff=0, color=RED),
                          Arrow(origin, origin + ground * SCALE, buff=0, color=YELLOW))

        def working(air):
            ground = air + wind
            parts = [VGroup(column(*air[:2], GREEN), Text("air", font_size=22, color=GREEN)).arrange(DOWN, buff=0.15),
                     Text("+", font_size=30),
                     VGroup(column(*wind[:2], RED), Text("wind", font_size=22, color=RED)).arrange(DOWN, buff=0.15),
                     Text("=", font_size=30),
                     VGroup(column(*ground[:2], YELLOW), Text("ground", font_size=22, color=YELLOW)).arrange(DOWN, buff=0.15)]
            return VGroup(*parts).arrange(RIGHT, buff=0.3).move_to([3, 2.6, 0])

        def caption(text):
            return Text(text, font_size=24).move_to([3, 0.9, 0])

        # 1. The nose points straight down the runway
        air = np.array([AIRSPEED, 0, 0])
        arrows, sums = diagram(air), working(air)
        message = caption("Nose straight down the runway...")
        plane = plane_shape().move_to(start)
        self.add(plane)
        self.play(GrowArrow(arrows[0]), FadeIn(sums[0]), Write(message))
        self.play(GrowArrow(arrows[1]), FadeIn(sums[1]), FadeIn(sums[2]))
        self.play(GrowArrow(arrows[2]), FadeIn(sums[3]), FadeIn(sums[4]))

        drift = np.array([3.5, 3.5 * -WIND / AIRSPEED, 0])
        track = DashedLine(start, start + drift, color=YELLOW)
        self.play(plane.animate.shift(drift), Create(track),
                  Transform(message, caption("...and the wind blows it off the runway")),
                  run_time=2.5, rate_func=linear)
        self.wait()

        # 2. The pilot turns the nose into the wind
        along = math.sqrt(AIRSPEED**2 - WIND**2)
        air = np.array([along, WIND, 0])
        angle = math.atan2(WIND, along)
        self.play(FadeOut(track), plane.animate.move_to(start))
        self.play(Transform(arrows, diagram(air)), Transform(sums, working(air)),
                  Rotate(plane, angle), Transform(message, caption(f"Nose turned {math.degrees(angle):.0f}° into the wind...")),
                  run_time=2)
        self.wait(0.5)

        track = DashedLine(start, start + np.array([11, 0, 0]), color=YELLOW)
        self.play(plane.animate.shift([11, 0, 0]), Create(track),
                  Transform(message, caption("...so air + wind points straight down the runway")),
                  run_time=4, rate_func=linear)
        self.wait()

Now experiment. Predict first each time, then run it.

1. Set `WIND = 0`. How far does the pilot need to turn the nose now?
2. Try `WIND = 80`, then `WIND = 120`. What happens to the angle, and to the yellow arrow's length?
3. What do you think happens if `WIND` is as big as `AIRSPEED`? Try it.
4. In the second half of the animation, the green arrow is longer than the yellow one. The plane flies through the air at 130 knots, so why does it move along the runway more slowly than that?

> ### Velocity is a vector
>
> **Speed** is just a number: 130 knots. **Velocity** is speed **and** direction, so it's a vector. Its length is the speed.
>
> When something moves through a medium that is itself moving, like a plane in wind or a boat in a river current, the velocities add:
>
> $$\text{velocity over the ground} = \text{velocity through the air} + \text{wind}$$
>
> To land, the pilot needs the **sum** to point down the runway. So the nose has to point partly **into** the wind, and part of the plane's speed is used up cancelling the wind.
>
> In the real videos, there's one more step: just before the wheels touch the ground, the pilot uses the rudder to swing the nose straight. That last-second swing is the dramatic bit.

GCSE Physics splits quantities into **scalars** (size only) and **vectors** (size and direction). Speed, distance, mass and time are scalars. Velocity, displacement and force are vectors.

Now use it: the plane flies through the air at 130 knots and the crosswind is 50 knots. When the nose is turned into the wind, the air velocity is $\begin{pmatrix} x \\ 50 \end{pmatrix}$ and its length is 130. Find $x$. That's how fast the plane moves down the runway. Then check your answer against the animation.

### Check yourself (no computer)

1. A plane flies at 100 knots through the air, and the crosswind is 60 knots. How fast does it move along the runway when it's crabbing?
2. A boat points straight across a river at $\begin{pmatrix} 0 \\ 4 \end{pmatrix}$ m/s, and the current flows at $\begin{pmatrix} 3 \\ 0 \end{pmatrix}$ m/s. What is the boat's actual velocity? What is its actual speed?
3. A plane's air velocity is $\begin{pmatrix} 150 \\ 0 \end{pmatrix}$ and the wind is $\begin{pmatrix} -30 \\ 0 \end{pmatrix}$. What is its velocity over the ground? What kind of wind is this?
4. Sort these into scalars and vectors: speed, velocity, distance, displacement, mass, force, time.

## Part 3: Vectors with 3 numbers

Everything so far has been flat: across and up. But the world has **three** dimensions. As well as across and up, you can move **forward and back**. A drone, a plane, or a CS player jumping off a box all need 3 numbers to say where they are. Every position on a CS map is a vector with 3 numbers.

So a 3D vector has a third number:

$$\begin{pmatrix} \text{across} \\ \text{up} \\ \text{forward} \end{pmatrix}$$

Game engines like Unity use this same order: across, then up, then forward.

The good news is that **everything you already know still works**. You just do it for one more number.

**Before you run it:** a drone takes off and flies 2 across, 3 up and 6 forward. Write its movement as a column vector. Then guess how far the drone is, in a straight line, from where it started.

In [ ]:
%%manim -ql -v WARNING --progress_bar none ThreeD
import math

# ✏️ Change these: across, up, forward
v = (2, 3, 6)


def fmt(n):
    return f"{round(n, 2):g}".replace("-", "−")


class ThreeD(ThreeDScene):
    def construct(self):
        across, up, forward = v
        axes = ThreeDAxes(x_range=[-7, 7], y_range=[-7, 7], z_range=[-5, 5],
                          x_length=14, y_length=14, z_length=10)

        def point(a, u, f):
            # Our order is (across, up, forward). manim's 3D axes are (across, forward, up)
            return axes.c2p(a, f, u)

        self.set_camera_orientation(phi=70 * DEGREES, theta=-50 * DEGREES, zoom=0.75)
        self.add(axes)

        # Label the axes. These labels always turn to face the camera
        for text, color, where in [("across", GREEN, point(7.6, 0, 0)),
                                   ("up", RED, point(0.8, 4.4, 0)),
                                   ("forward", BLUE, point(0, 0, 7.8))]:
            self.add_fixed_orientation_mobjects(Text(text, font_size=28, color=color).move_to(where))

        # The steps, one at a time: across (green), then up (red), then forward (blue)
        corners = [point(0, 0, 0), point(across, 0, 0), point(across, up, 0), point(across, up, forward)]
        for start, end, color in zip(corners, corners[1:], [GREEN, RED, BLUE]):
            if np.linalg.norm(end - start) > 0.01:
                self.play(GrowFromPoint(Arrow3D(start, end, color=color), start))

        # The total movement, straight there (yellow)
        if (across, up, forward) != (0, 0, 0):
            self.play(GrowFromPoint(Arrow3D(corners[0], corners[3], color=YELLOW), corners[0]))

        # The length, in one go: the same rule as in 2D, with one more number
        total_sq = across**2 + up**2 + forward**2
        length = math.sqrt(total_sq)
        answer = f" = {fmt(length)}" if length == int(length) else f" ≈ {length:.2f}"
        column = Text(f"({fmt(across)}, {fmt(up)}, {fmt(forward)}):  "
                      f"{fmt(across)} across, {fmt(up)} up, {fmt(forward)} forward", font_size=24)
        length_text = Text(f"length = √({fmt(across)}² + {fmt(up)}² + {fmt(forward)}²) = √{fmt(total_sq)}{answer}",
                           font_size=24, color=YELLOW)
        rule = Text("(the same rule as 2D, with one more number)", font_size=20, color=GREY_B)
        panel = VGroup(column, length_text, rule).arrange(DOWN, aligned_edge=LEFT)
        panel.to_corner(UL).add_background_rectangle(opacity=0.85, buff=0.2)
        self.add_fixed_in_frame_mobjects(panel)
        self.remove(panel)

        self.play(FadeIn(panel.background_rectangle), Write(column))
        self.play(Write(length_text))
        self.play(FadeIn(rule))

        # Spin the camera round to see that it really is 3D
        self.begin_ambient_camera_rotation(rate=0.35)
        self.wait(6)
        self.stop_ambient_camera_rotation()

Now experiment. Predict first each time, then run it.

1. Try `v = (4, 0, 3)`. What kind of vector is this really? What's its length?
2. Predict the length of `(1, 2, 2)` on paper, then check.
3. Try `v = (-2, 3, -6)`. Is it the same length as `(2, 3, 6)`? Why?

> ### Definition: vectors in 3D
>
> A 3D vector has three numbers: $\begin{pmatrix} \text{across} \\ \text{up} \\ \text{forward} \end{pmatrix}$. Everything works exactly like in 2D, just with one more number:
>
> - **Adding and subtracting:** do it to each number. $\begin{pmatrix} 1 \\ 2 \\ 3 \end{pmatrix} + \begin{pmatrix} 4 \\ 0 \\ -1 \end{pmatrix} = \begin{pmatrix} 5 \\ 2 \\ 2 \end{pmatrix}$
> - **Multiplying by a number:** multiply each number. $2\begin{pmatrix} 1 \\ 2 \\ 3 \end{pmatrix} = \begin{pmatrix} 2 \\ 4 \\ 6 \end{pmatrix}$
> - **Negatives, equal vectors and the zero vector** $\begin{pmatrix} 0 \\ 0 \\ 0 \end{pmatrix}$ all mean the same as before.
> - **Length:** the same rule as in 2D. Square each number, add them up, then square root:
>
> $$\left|\begin{pmatrix} x \\ y \\ z \end{pmatrix}\right| = \sqrt{x^2 + y^2 + z^2}$$

### Why does it work? (The GCSE way)

In 2D, the length came from **one** right-angled triangle. In 3D there are **two**, and you use Pythagoras on each in turn. This time the drone flies across, then forward along the floor, then up, so you can see both triangles.

**Step 1: the floor.** Across and forward make a right-angled triangle lying flat on the floor. Its long side is the **floor diagonal**:

$$\text{floor}^2 = 2^2 + 6^2 = 40, \quad \text{so the floor diagonal is } \sqrt{40}$$

Leave it as $\sqrt{40}$. Don't round it, because you're about to square it again.

**Step 2: the length.** The floor diagonal and the up step make a second right-angled triangle, standing upright. Its long side is the yellow arrow:

$$\text{length}^2 = \text{floor}^2 + 3^2 = 40 + 9 = 49, \quad \text{so the length is } \sqrt{49} = 7$$

GCSE questions about the diagonal of a box are usually marked this way, so show both steps in an exam.

**Like a normal human:** look at step 2 again. $\text{floor}^2$ was just $2^2 + 6^2$, so $\text{length}^2 = 2^2 + 6^2 + 3^2$. The floor was only a stepping stone. If you just want the length, skip it and do everything in one go:

$$\sqrt{2^2 + 3^2 + 6^2} = \sqrt{49} = 7$$

Same answer, one line. That's the rule in the definition box.

In [ ]:
%%manim -ql -v WARNING --progress_bar none TwoSteps
import math

# ✏️ Change these: across, up, forward
v = (2, 3, 6)


def fmt(n):
    return f"{round(n, 2):g}".replace("-", "−")


class TwoSteps(ThreeDScene):
    def construct(self):
        across, up, forward = v
        axes = ThreeDAxes(x_range=[-7, 7], y_range=[-7, 7], z_range=[-5, 5],
                          x_length=14, y_length=14, z_length=10)

        def point(a, u, f):
            # Our order is (across, up, forward). manim's 3D axes are (across, forward, up)
            return axes.c2p(a, f, u)

        def grow(start, end, color):
            # Draw an arrow, unless it would have zero length
            if np.linalg.norm(end - start) > 0.01:
                self.play(GrowFromPoint(Arrow3D(start, end, color=color), start))

        def dashed(start, end):
            if np.linalg.norm(end - start) > 0.01:
                self.play(Create(DashedLine(start, end, color=WHITE)))

        self.set_camera_orientation(phi=70 * DEGREES, theta=-50 * DEGREES, zoom=0.75)
        self.add(axes)
        for text, color, where in [("across", GREEN, point(7.6, 0, 0)),
                                   ("up", RED, point(0.8, 4.4, 0)),
                                   ("forward", BLUE, point(0, 0, 7.8))]:
            self.add_fixed_orientation_mobjects(Text(text, font_size=28, color=color).move_to(where))

        origin = point(0, 0, 0)
        floor_corner = point(across, 0, forward)
        tip = point(across, up, forward)

        # The working, fixed to the screen
        floor_sq = across**2 + forward**2
        total_sq = floor_sq + up**2
        length = math.sqrt(total_sq)
        answer = f" = {fmt(length)}" if length == int(length) else f" ≈ {length:.2f}"
        step1 = Text(f"Step 1, floor:    √({fmt(across)}² + {fmt(forward)}²) = √{fmt(floor_sq)}", font_size=24)
        step2 = Text(f"Step 2, length: √({fmt(floor_sq)} + {fmt(up)}²) = √{fmt(total_sq)}{answer}",
                     font_size=24, color=YELLOW)
        panel = VGroup(step1, step2).arrange(DOWN, aligned_edge=LEFT)
        panel.to_corner(UL).add_background_rectangle(opacity=0.85, buff=0.2)
        self.add_fixed_in_frame_mobjects(panel)
        self.remove(panel)

        # Triangle 1, flat on the floor: across, then forward, then the floor diagonal
        grow(origin, point(across, 0, 0), GREEN)
        grow(point(across, 0, 0), floor_corner, BLUE)
        self.play(FadeIn(panel.background_rectangle))
        dashed(origin, floor_corner)
        self.play(Write(step1))

        # Triangle 2, standing up: the floor diagonal, then up, then the total length
        grow(floor_corner, tip, RED)
        grow(origin, tip, YELLOW)
        self.play(Write(step2))

        self.begin_ambient_camera_rotation(rate=0.35)
        self.wait(6)
        self.stop_ambient_camera_rotation()

3D vectors aren't on your GCSE, but **3D Pythagoras is**: finding the longest diagonal of a box is a Higher tier question, and it's exactly the calculation in the second animation.

Now use it:

1. A box is 2 m wide, 3 m tall and 6 m long. How long is the longest stick that fits inside it, from one corner to the opposite corner? Do it the GCSE way (floor first), then check it in one go.
2. In a game, player A stands at $\begin{pmatrix} 10 \\ 0 \\ 4 \end{pmatrix}$ and player B is on a ledge at $\begin{pmatrix} 13 \\ 4 \\ 16 \end{pmatrix}$. Find the vector from A to B, then the straight-line distance between them. *(Hint: it's the same "tip to tip" subtraction as in session 2.)*

### Check yourself (no computer)

1. Work out $\begin{pmatrix} 1 \\ 4 \\ -2 \end{pmatrix} + \begin{pmatrix} 3 \\ -1 \\ 5 \end{pmatrix}$.
2. Work out $3\begin{pmatrix} 2 \\ 0 \\ -1 \end{pmatrix}$.
3. Find the length of $\begin{pmatrix} 2 \\ 6 \\ 9 \end{pmatrix}$.
4. Which is longer: $\begin{pmatrix} 1 \\ 1 \\ 1 \end{pmatrix}$ or $\begin{pmatrix} 0 \\ 0 \\ 2 \end{pmatrix}$?

## Coming up

You've now used vectors with 2 numbers and with 3 numbers. Next, we'll jump to vectors with **100 numbers**, which is how AI stores the meaning of a word. Nobody can draw 100 dimensions, but the maths is exactly the same: add them, subtract them, and measure their length. The length rule doesn't change either: square all 100 numbers, add them up, then square root.